# Chapter 16 — Simultaneous Equations Models
## University of Neuchâtel — Python for Econometrics

Topics:
1. Simultaneity bias — why OLS fails
2. Identification: order and rank conditions
3. Reduced form equations
4. 2SLS for simultaneous systems

**Format:** One cell per step. Manual calculation first, then statsmodels/linearmodels, then `np.isclose()`.

## 0 — Setup

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
import statsmodels.api as sm
import matplotlib.pyplot as plt
import wooldridge as woo
import os
try:
    from linearmodels.iv import IV2SLS
    HAS_LM = True
except ImportError:
    HAS_LM = False

In [ ]:
chapter_dir = os.path.dirname(os.path.abspath('__file__'))
os.chdir(chapter_dir)
output_dir = os.path.join(chapter_dir, 'output')
os.makedirs(output_dir, exist_ok=True)
print('Working directory:', chapter_dir)

## 1 — Simultaneity Problem

**Classic example:** Supply and demand.

**Demand:** $Q_d = \alpha_0 + \alpha_1 P + \alpha_2 Income + u_d$

**Supply:** $Q_s = \beta_0 + \beta_1 P + \beta_2 Input\_cost + u_s$

Equilibrium: $Q_d = Q_s = Q$

**Problem:** $P$ is determined by BOTH equations — it's correlated with both $u_d$ and $u_s$.
OLS of demand OR supply gives biased estimates.

**Solution:** 2SLS using instruments from each equation.

## 2 — Simulate Supply & Demand Data

In [ ]:
np.random.seed(123)
N = 500
# Exogenous variables
income     = np.random.normal(50, 10, N)   # demand shifter
input_cost = np.random.normal(20, 5, N)    # supply shifter
# True structural parameters
alpha_0, alpha_1, alpha_2 = 100, -2, 0.5  # demand
beta_0,  beta_1,  beta_2  = 10,  3, -1.5  # supply
# Structural errors
ud = np.random.normal(0, 5, N)
us = np.random.normal(0, 5, N)
# Reduced form: solve for P and Q
# alpha_1*P + alpha_2*income + ud = beta_1*P + beta_2*input_cost + us
# P = (beta_0 - alpha_0 + beta_2*input_cost - alpha_2*income + us - ud) / (alpha_1 - beta_1)
denom = alpha_1 - beta_1
P = (beta_0 - alpha_0 + beta_2*input_cost - alpha_2*income + us - ud) / denom
Q = alpha_0 + alpha_1*P + alpha_2*income + ud
df_sd = pd.DataFrame({'Q': Q, 'P': P, 'income': income, 'input_cost': input_cost})
print('Supply-Demand simulation:')
print(df_sd.describe().round(2))

In [ ]:
fig, ax = plt.subplots(figsize=(7,5))
ax.scatter(df_sd['P'], df_sd['Q'], alpha=0.3, s=15)
ax.set_xlabel('Price')
ax.set_ylabel('Quantity')
ax.set_title('Equilibrium (P, Q) observations\n(NOT a demand curve!)')
plt.tight_layout()
plt.savefig(os.path.join(output_dir,'fig_ch16_01_supply_demand.png'), dpi=150, bbox_inches='tight')
plt.show()

## 3 — Identification Conditions

**Order condition (necessary):** An equation is identified if the number of excluded
exogenous variables $\geq$ number of included endogenous variables − 1.

**Demand equation:** includes $P$ (endog), $income$ (exog). Excluded: $input\_cost$.
Order condition: excluded (1) ≥ endog − 1 (1−1=0) → **identified** ✓

**Supply equation:** includes $P$ (endog), $input\_cost$ (exog). Excluded: $income$.
Order condition: excluded (1) ≥ 1−1=0 → **identified** ✓

## 4 — Naive OLS (Simultaneity Bias Demo)

### Step 4.1 — OLS of demand equation (biased)

In [ ]:
n_sd = len(df_sd)
X_demand_ols = np.column_stack([np.ones(n_sd), df_sd['P'].values, df_sd['income'].values])
y_Q = df_sd['Q'].values
beta_demand_ols = np.linalg.solve(X_demand_ols.T @ X_demand_ols, X_demand_ols.T @ y_Q)
print('OLS of demand equation (biased):')
print(f'  alpha_0 (true={alpha_0}): {beta_demand_ols[0]:.3f}')
print(f'  alpha_1 (true={alpha_1}): {beta_demand_ols[1]:.3f}  ← biased!')
print(f'  alpha_2 (true={alpha_2}): {beta_demand_ols[2]:.3f}')

### Step 4.2 — Statsmodels OLS demand

In [ ]:
res_ols_d = smf.ols('Q ~ P + income', data=df_sd).fit()
print(res_ols_d.summary())
print(f'True alpha_1={alpha_1}, OLS gives {res_ols_d.params["P"]:.3f}')

## 5 — Reduced Form Equations

**Reduced form:** Express each endogenous variable as a function of ALL exogenous variables only.

**RF for P:**

$$P = \pi_{10} + \pi_{11} \cdot income + \pi_{12} \cdot input\_cost + v_1$$

**RF for Q:**

$$Q = \pi_{20} + \pi_{21} \cdot income + \pi_{22} \cdot input\_cost + v_2$$

Each RF can be estimated by OLS — no simultaneity bias.

### Step 5.1 — Reduced form for P (manual + statsmodels + compare)

In [ ]:
X_rf = np.column_stack([np.ones(n_sd), df_sd['income'].values, df_sd['input_cost'].values])
pi_P = np.linalg.solve(X_rf.T @ X_rf, X_rf.T @ df_sd['P'].values)
print('RF for P (manual):', pi_P.round(4))

In [ ]:
res_rf_P = smf.ols('P ~ income + input_cost', data=df_sd).fit()
print(res_rf_P.summary())

In [ ]:
for i, lbl in enumerate(['const','income','input_cost']):
    b_sm = res_rf_P.params[lbl]
    print(f'  {lbl:12s}: manual={pi_P[i]:.4f}, sm={b_sm:.4f}, match={np.isclose(pi_P[i],b_sm)}')

### Step 5.2 — Reduced form for Q (manual + statsmodels + compare)

In [ ]:
pi_Q = np.linalg.solve(X_rf.T @ X_rf, X_rf.T @ y_Q)
res_rf_Q = smf.ols('Q ~ income + input_cost', data=df_sd).fit()
print('RF for Q (manual):', pi_Q.round(4))
for i, lbl in enumerate(['const','income','input_cost']):
    b_sm = res_rf_Q.params[lbl]
    print(f'  {lbl:12s}: manual={pi_Q[i]:.4f}, sm={b_sm:.4f}, match={np.isclose(pi_Q[i],b_sm)}')

## 6 — 2SLS for Demand Equation

### Step 6.1 — Stage 1: P ~ income + input_cost (fitted P_hat)

In [ ]:
P_hat = res_rf_P.fittedvalues.values
df_sd['P_hat'] = P_hat
print('P vs P_hat (first 5 rows):')
print(df_sd[['P','P_hat']].head().round(3))

### Step 6.2 — Stage 2: Q ~ P_hat + income (manual)

In [ ]:
X_s2 = np.column_stack([np.ones(n_sd), P_hat, df_sd['income'].values])
beta_2sls_demand = np.linalg.solve(X_s2.T @ X_s2, X_s2.T @ y_Q)
labels_d = ['alpha_0','alpha_1_P','alpha_2_income']
print('2SLS demand equation (manual):')
for lbl, b, true_b in zip(labels_d, beta_2sls_demand, [alpha_0, alpha_1, alpha_2]):
    print(f'  {lbl:18s}: estimated={b:.3f},  true={true_b}')

### Step 6.3 — 2SLS with linearmodels IV2SLS

In [ ]:
if HAS_LM:
    dep_d   = df_sd['Q']
    exog_d  = sm.add_constant(df_sd['income'])
    endog_d = df_sd[['P']]
    instr_d = df_sd[['input_cost']]
    res_iv_d = IV2SLS(dep_d, exog_d, endog_d, instr_d).fit(cov_type='unadjusted')
    print(res_iv_d.summary)
else:
    print('Using manual 2SLS result above')

### Step 6.4 — Compare 2SLS manual vs IV2SLS vs true params

In [ ]:
if HAS_LM:
    iv_params = res_iv_d.params
    match_P = np.isclose(beta_2sls_demand[1], iv_params['P'])
    print(f'alpha_1 (P): manual={beta_2sls_demand[1]:.4f}, IV2SLS={iv_params["P"]:.4f}, match={match_P}')
    print(f'True alpha_1 = {alpha_1}')
print('\n2SLS vs OLS vs true:')
print(f'  OLS:     alpha_1 = {res_ols_d.params["P"]:.3f}  (biased, true={alpha_1})')
print(f'  2SLS:    alpha_1 = {beta_2sls_demand[1]:.3f}  (consistent, true={alpha_1})')

## 7 — Application: Labour Supply (mroz dataset)

In [ ]:
mroz = woo.dataWoo('mroz')
mroz = mroz[mroz['inlf']==1].copy()  # restrict to women in labour force
mroz['exper2'] = mroz['exper']**2
print('mroz (women in LF), shape:', mroz.shape)
print(mroz[['hours','lwage','educ','exper','kidslt6','nwifeinc']].head(8))

In [ ]:
# OLS labour supply equation: hours ~ lwage + kidslt6 + kidsge6 + nwifeinc
res_hours_ols = smf.ols('hours ~ lwage + kidslt6 + kidsge6 + nwifeinc', data=mroz).fit()
print('OLS labour supply (biased — lwage is endogenous):')
print(res_hours_ols.params.round(4))

In [ ]:
# 2SLS: instrument for lwage = educ, exper, exper2
if HAS_LM:
    mroz_clean = mroz[['hours','lwage','kidslt6','kidsge6','nwifeinc','educ','exper','exper2']].dropna()
    dep_h   = mroz_clean['hours']
    exog_h  = sm.add_constant(mroz_clean[['kidslt6','kidsge6','nwifeinc']])
    endog_h = mroz_clean[['lwage']]
    instr_h = mroz_clean[['educ','exper','exper2']]
    res_iv_h = IV2SLS(dep_h, exog_h, endog_h, instr_h).fit(cov_type='unadjusted')
    print('2SLS labour supply:')
    print(res_iv_h.summary)
else:
    # Manual 2SLS
    mroz_clean = mroz.dropna()
    res_s1_h = smf.ols('lwage ~ educ + exper + exper2', data=mroz_clean).fit()
    mroz_clean = mroz_clean.copy()
    mroz_clean['lwage_hat'] = res_s1_h.fittedvalues
    res_s2_h = smf.ols('hours ~ lwage_hat + kidslt6 + kidsge6 + nwifeinc', data=mroz_clean).fit()
    print('2SLS labour supply (manual):')
    print(res_s2_h.params.round(4))

## 8 — Summary Table

In [ ]:
sumtab = pd.DataFrame({
    'Model': ['Demand OLS (sim)','Demand 2SLS (sim)','Hours OLS (mroz)'],
    'P_or_lwage_coeff': [
        res_ols_d.params['P'],
        beta_2sls_demand[1],
        res_hours_ols.params['lwage']
    ]
})
print(sumtab.round(4).to_string(index=False))
sumtab.to_csv(os.path.join(output_dir,'table_ch16_sem_summary.csv'), index=False)
print('Saved table_ch16_sem_summary.csv')